<!-- open-in-badges -->
[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EAFIT-IA/si7011-DeepLearning/blob/main/sessions/04_representation/notebooks/sesion_04_4_simclr.ipynb) [![Abrir en Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/EAFIT-IA/si7011-DeepLearning/blob/main/sessions/04_representation/notebooks/sesion_04_4_simclr.ipynb) [![Abrir en Lightning Studio](https://pl-bolts-doc-images.s3.us-east-2.amazonaws.com/app-2/studio-badge.svg)](https://lightning.ai/new?repo_url=https%3A%2F%2Fgithub.com%2FEAFIT-IA%2Fsi7011-DeepLearning%2Fblob%2Fmain%2Fsessions%2F04_representation%2Fnotebooks%2Fsesion_04_4_simclr.ipynb)

# SI7011 · Sesión 04 · 4. Aprendizaje contrastivo: SimCLR
**Juan David Martínez Vargas · Universidad EAFIT**

Acompaña la **Parte 4** de las diapositivas. Ya no hay decoder: la red no redibuja píxeles. Aprende a reconocer que dos versiones alteradas de una imagen son **la misma imagen**, distintas de todas las demás del lote.

| Sección | Lo que cambia | Qué mirar |
|---|---|---|
| 1 | dos vistas de cada imagen | qué invariancias le pedimos a la red |
| 2 | $g$, $p$ y la pérdida InfoNCE | es una entropía cruzada sobre $2N-1$ candidatos |
| 3 | 3 épocas en vivo | la pérdida baja desde $\ln(2N-1)$ |
| 4 | el *checkpoint* largo, ya entrenado | $k$-NN sobre $\mathbf h$ sin haber visto etiquetas |
| 5 | la matriz de similitudes | la figura de la diapositiva |
| 6 | sin *color jitter* | el atajo del color |

**Datos:** CIFAR-10, 50 000 imágenes de 32×32 a color; las etiquetas solo se usan para **medir**. **Usa GPU** (Colab o Kaggle): la sección 3 tarda unos 3 minutos. **En Kaggle:** agrega con **Add Input** los datasets [`pankrzysiu/cifar10-python`](https://www.kaggle.com/datasets/pankrzysiu/cifar10-python) y [`juanmartinezv4399/si7011-s04-simclr`](https://www.kaggle.com/datasets/juanmartinezv4399/si7011-s04-simclr). Un entrenamiento largo toma horas, así que en clase cargamos un *checkpoint* ya entrenado (sección 4).

In [ ]:
import json, math, time
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from torch import nn
from torchvision import datasets
from torchvision.models import resnet18

torch.manual_seed(0)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': 0.3})
print('PyTorch', torch.__version__, '· device', device)

## Datos: CIFAR-10 en la GPU
Las 60 000 imágenes ocupan 180 MB como enteros de 8 bits, así que caben enteras en la GPU. Los aumentos también se hacen en la GPU, sobre lotes completos: con dos núcleos de CPU (Colab, Kaggle), aumentar imagen por imagen sería el cuello de botella.

### Cargar CIFAR-10
La celda busca los datos en este orden y usa el primero que encuentre. Imprime la **fuente** usada: revísala.

| Plataforma | De dónde salen los datos |
|---|---|
| **Kaggle** | el dataset [`pankrzysiu/cifar10-python`](https://www.kaggle.com/datasets/pankrzysiu/cifar10-python) agregado con **Add Input** (`/kaggle/input/...`), sin descargar |
| **Lightning Studio** o **local** | la carpeta `data/` junto al notebook, si ya tiene los archivos de una ejecución anterior |
| **Colab** | `kagglehub` con el mismo dataset y tu secreto `KAGGLE_API_TOKEN` (el mismo de la sesión 01) |
| sin dataset ni token | descarga del sitio original con `torchvision` (unos 170 MB; puede tardar varios minutos) |

Es el mismo cargador de la sesión 03: cinco lotes de entrenamiento y uno de prueba, cada uno con 10 000 imágenes guardadas como filas de 3 072 bytes (R, luego G, luego B).

In [ ]:
import os, pickle, tarfile
from pathlib import Path

KAGGLE_HANDLE = 'pankrzysiu/cifar10-python'          # CIFAR-10 en el formato original para Python
ROOTS = [Path('data'), Path('/kaggle/input'), Path('/teamspace/studios/this_studio/data')]

def kaggle_token():
    # Colab: secreto KAGGLE_API_TOKEN. Local o Lightning: variable de entorno o ~/.kaggle/access_token.
    if 'KAGGLE_API_TOKEN' not in os.environ:
        try:
            from google.colab import userdata
            os.environ['KAGGLE_API_TOKEN'] = userdata.get('KAGGLE_API_TOKEN').strip()
        except Exception:
            pass
    return 'KAGGLE_API_TOKEN' in os.environ or (Path.home() / '.kaggle' / 'access_token').exists()

def find_batches(roots):
    for root in [r for r in roots if r.exists()]:
        hit = next(root.rglob('data_batch_1'), None)
        if hit is None:                                    # algunas copias traen solo el comprimido
            tgz = next(root.rglob('cifar-10-python.tar.gz'), None)
            if tgz is not None:
                print('Descomprimiendo', tgz, '→ data/')
                with tarfile.open(tgz) as t:
                    t.extractall('data')
                hit = next(Path('data').rglob('data_batch_1'), None)
        if hit is not None:
            return hit.parent
    return None

def locate_cifar10():
    folder = find_batches(ROOTS)
    if folder is not None:
        return folder, 'input de Kaggle' if str(folder).startswith('/kaggle/input') else 'archivos locales'
    if kaggle_token() or Path('/kaggle/input').exists():
        try:
            import kagglehub                               # preinstalado en Colab y Kaggle
            folder = find_batches([Path(kagglehub.dataset_download(KAGGLE_HANDLE))])
            if folder is not None:
                return folder, 'kagglehub'
        except Exception as e:
            print('kagglehub no disponible:', type(e).__name__, '→ se descarga del sitio original')
    datasets.CIFAR10('data', download=True)                # el mismo formato, en data/cifar-10-batches-py
    return find_batches([Path('data')]), 'torchvision (sitio original)'

def read_batch(path):
    with open(path, 'rb') as f:
        d = pickle.load(f, encoding='bytes')
    # cada fila: 3 072 bytes = R (1 024) | G (1 024) | B (1 024) → 3 × 32 × 32 → H × W × C, como un PNG
    x = np.asarray(d[b'data'], dtype=np.uint8).reshape(-1, 3, 32, 32).transpose(0, 2, 3, 1)
    return x, np.asarray(d[b'labels'], dtype=np.int64)

def load_cifar10():
    folder, source = locate_cifar10()
    parts = [read_batch(folder / f'data_batch_{i}') for i in range(1, 6)]   # 5 × 10 000 de entrenamiento
    x_test, y_test = read_batch(folder / 'test_batch')                      # 10 000 de prueba
    with open(folder / 'batches.meta', 'rb') as f:
        classes = [c.decode() for c in pickle.load(f, encoding='bytes')[b'label_names']]
    print('Fuente:', source, '·', folder)
    return np.concatenate([x for x, _ in parts]), np.concatenate([y for _, y in parts]), x_test, y_test, classes

x_train_raw, y_train_raw, x_test_raw, y_test_raw, CLASSES = load_cifar10()
x_train_raw.shape, x_test_raw.shape

In [ ]:
X_train = torch.from_numpy(x_train_raw).permute(0, 3, 1, 2).to(device)    # uint8, N × 3 × 32 × 32
X_test = torch.from_numpy(x_test_raw).permute(0, 3, 1, 2).to(device)
y_train, y_test = torch.from_numpy(y_train_raw), torch.from_numpy(y_test_raw)   # solo para medir
MEAN = torch.tensor([0.4914, 0.4822, 0.4465], device=device).view(1, 3, 1, 1)
STD = torch.tensor([0.2470, 0.2435, 0.2616], device=device).view(1, 3, 1, 1)
norm = lambda x: (x - MEAN) / STD                                          # x ∈ [0, 1] → normalizado
X_train.shape, X_test.shape

## 1. Dos vistas de cada imagen
`vista(x)` aplica, con parámetros aleatorios **distintos para cada imagen**, la receta de SimCLR para CIFAR-10 (diapositiva *Augmentations define "the same"*):

1. recorte aleatorio de entre el 20 % y el 100 % del área, reescalado a 32×32, y espejo horizontal con probabilidad ½;
2. *color jitter* con probabilidad 0.8: brillo, contraste y saturación ×[0.6, 1.4], tono ±0.1 vueltas;
3. escala de grises con probabilidad 0.2.

Dos llamadas a `vista` sobre el mismo lote dan los pares positivos $(\tilde{\mathbf x}_i, \tilde{\mathbf x}_j)$. Lo que estas transformaciones cambian es justo lo que $\mathbf h$ aprenderá a **ignorar**.

In [ ]:
def recorte_espejo(x, escala=(0.2, 1.0), razon=(3 / 4, 4 / 3)):
    # recorte aleatorio reescalado + espejo, con una transformación afín por imagen (grid_sample)
    B = len(x)
    area = torch.empty(B, device=x.device).uniform_(*escala)
    r = torch.exp(torch.empty(B, device=x.device).uniform_(math.log(razon[0]), math.log(razon[1])))
    w, h = (area * r).sqrt().clamp(max=1), (area / r).sqrt().clamp(max=1)     # fracción del ancho y del alto
    cx = (torch.rand(B, device=x.device) * 2 - 1) * (1 - w)                    # centro, en coordenadas [−1, 1]
    cy = (torch.rand(B, device=x.device) * 2 - 1) * (1 - h)
    espejo = torch.where(torch.rand(B, device=x.device) < 0.5, -1.0, 1.0)
    theta = torch.zeros(B, 2, 3, device=x.device)
    theta[:, 0, 0], theta[:, 0, 2], theta[:, 1, 1], theta[:, 1, 2] = w * espejo, cx, h, cy
    grid = F.affine_grid(theta, x.shape, align_corners=False)
    return F.grid_sample(x, grid, mode='bilinear', padding_mode='reflection', align_corners=False)

def gris(x):
    return (0.299 * x[:, :1] + 0.587 * x[:, 1:2] + 0.114 * x[:, 2:]).expand_as(x)

def color_jitter(x, b=0.4, c=0.4, s=0.4, hue=0.1, p=0.8):
    B = len(x); u = lambda a: torch.empty(B, 1, 1, 1, device=x.device).uniform_(1 - a, 1 + a)
    y = x * u(b)                                                                   # brillo
    m = gris(y).mean((1, 2, 3), keepdim=True); y = (y - m) * u(c) + m              # contraste
    gy = gris(y); y = (y - gy) * u(s) + gy                                         # saturación
    # tono: rotación en el plano de color (I, Q) del espacio YIQ
    yiq = torch.tensor([[0.299, 0.587, 0.114], [0.596, -0.274, -0.322], [0.211, -0.523, 0.312]], device=x.device)
    ang = torch.empty(B, device=x.device).uniform_(-hue, hue) * 2 * math.pi
    rot = torch.zeros(B, 3, 3, device=x.device); rot[:, 0, 0] = 1
    rot[:, 1, 1], rot[:, 1, 2], rot[:, 2, 1], rot[:, 2, 2] = ang.cos(), -ang.sin(), ang.sin(), ang.cos()
    M = torch.linalg.inv(yiq) @ rot @ yiq                                          # B × 3 × 3
    y = torch.einsum('bij,bjhw->bihw', M, y).clamp(0, 1)
    aplicar = (torch.rand(B, 1, 1, 1, device=x.device) < p)
    return torch.where(aplicar, y, x)

def vista(xb, color=True):
    # x̃ = t(x), t ∼ 𝒯: una vista aleatoria por imagen; uint8 → normalizada
    x = recorte_espejo(xb.float() / 255)
    if color:
        x = color_jitter(x)
        x = torch.where(torch.rand(len(x), 1, 1, 1, device=x.device) < 0.2, gris(x), x)
    return norm(x.clamp(0, 1))

In [ ]:
# 8 imágenes, una por clase, y dos vistas de cada una
torch.manual_seed(3)
idx8 = torch.stack([(y_train == c).nonzero()[0, 0] for c in range(8)])
xb = X_train[idx8]
v1, v2 = vista(xb), vista(xb)
ver = lambda t: (t * STD + MEAN).clamp(0, 1).permute(0, 2, 3, 1).cpu()
fig, axes = plt.subplots(3, 8, figsize=(10, 4))
for c in range(8):
    for r, im in enumerate([xb.permute(0, 2, 3, 1).cpu() / 255, ver(v1), ver(v2)]):
        axes[r, c].imshow(im[c]); axes[r, c].axis('off')
for r, n in enumerate(['x', 'vista 1', 'vista 2']):
    axes[r, 0].set_title(n, fontsize=8, loc='left')
plt.tight_layout(); plt.show()
np.savez_compressed('vistas_ejemplo.npz', x=xb.cpu().numpy(), v1=ver(v1).numpy(), v2=ver(v2).numpy())   # figura S04-F05

Algunas vistas apenas se parecen: un recorte pequeño de la rueda de un camión, en gris, frente al camión entero con otro tono. Para la red, deben producir el mismo $\mathbf u$.

## 2. El encoder $g$, la cabeza $p$ y la pérdida InfoNCE
- $g$: la ResNet-18 de la sesión 03, adaptada a imágenes de 32×32 (primera convolución de 3×3 sin *stride* y sin *max pool*), sin la capa final. $\mathbf h = g(\tilde{\mathbf x}) \in \mathbb R^{512}$.
- $p$: Linear–BN–ReLU–Linear, de 512 a 128. $\mathbf u = p(\mathbf h)$ (diapositiva *Why a projection head?*).

La pérdida de la diapositiva *InfoNCE is a cross-entropy*:
$$s_{ij} = \frac{\mathbf u_i^\top \mathbf u_j}{\tau\,\lVert\mathbf u_i\rVert\,\lVert\mathbf u_j\rVert}, \qquad L_i = -\log \frac{\exp(s_{i,j(i)})}{\sum_{k \ne i} \exp(s_{ik})}$$
Las $2N$ vistas se apilan como `[vista 1 de las N imágenes, vista 2 de las N imágenes]`, así que la pareja de la fila $i$ está en la columna $i + N$ (o $i - N$). Cada fila de la matriz $S$ es un problema de clasificación con $2N - 1$ clases, y la clase correcta es la otra vista.

In [ ]:
def resnet18_cifar():
    m = resnet18(weights=None)
    m.conv1 = nn.Conv2d(3, 64, 3, stride=1, padding=1, bias=False)   # 32×32 no se reduce al entrar
    m.maxpool = nn.Identity()
    m.fc = nn.Identity()                                              # g(x) = h ∈ ℝ^512
    return m

def cabeza():
    return nn.Sequential(nn.Linear(512, 512), nn.BatchNorm1d(512), nn.ReLU(), nn.Linear(512, 128))   # u = p(h) ∈ ℝ^128

def info_nce(u, tau):
    u = F.normalize(u, dim=1)                          # u / ‖u‖: el producto punto queda como coseno
    S = u @ u.T / tau                                  # s_ij para las 2N × 2N parejas
    S = S.masked_fill(torch.eye(len(u), dtype=torch.bool, device=u.device), float('-inf'))   # k ≠ i
    pos = torch.arange(len(u), device=u.device).roll(len(u) // 2)                           # j(i): la otra vista
    loss = F.cross_entropy(S, pos)                     # promedio de L_i = −log softmax(S_i)_{j(i)}
    acierto = (S.argmax(1) == pos).float().mean()      # ¿el más parecido es la otra vista?
    return loss, acierto

Comprobamos que `F.cross_entropy` calcula exactamente la fórmula, con un lote pequeño de $N = 4$ y vectores aleatorios.

In [ ]:
torch.manual_seed(0)
u, tau = torch.randn(8, 128), 0.2
loss, _ = info_nce(u, tau)
un = F.normalize(u, dim=1); S = un @ un.T / tau
L = [-(S[i, (i + 4) % 8] - torch.logsumexp(S[i][torch.arange(8) != i], 0)) for i in range(8)]   # la fórmula, fila por fila
print(f'info_nce {loss:.6f} · fórmula {torch.stack(L).mean():.6f}')
print(f'con u aleatorios, L ≈ ln(2N − 1) = ln 7 = {math.log(7):.3f}')

## 3. Tres épocas en vivo
$N = 512$ imágenes por lote, es decir $2N = 1024$ vistas y 1023 candidatos por fila. SGD con *momentum*, $\eta$ con *warmup* y coseno (sesión 02), $\tau = 0.2$, y precisión mixta en la GPU.

**Predice:** al empezar, ¿qué valor tendrá la pérdida? (pista: la última línea de la celda anterior, con $2N = 1024$).

In [ ]:
N, TAU = 512, 0.2
EPOCAS_VIVO = 3
amp = device.type == 'cuda'

def entrenar(g, p, epocas, color=True, lr=0.06 * N / 256, wd=5e-4):
    params = [*g.parameters(), *p.parameters()]
    opt = torch.optim.SGD(params, lr=lr, momentum=0.9, weight_decay=wd, nesterov=True)
    pasos_ep = len(X_train) // N
    total, warm = epocas * pasos_ep, max(1, epocas * pasos_ep // 10)       # warmup: el primer 10 % de los pasos
    f = lambda t: (t + 1) / warm if t < warm else 0.5 * (1 + math.cos(math.pi * (t - warm) / max(1, total - warm)))
    sched = torch.optim.lr_scheduler.LambdaLR(opt, f)
    scaler = torch.amp.GradScaler(enabled=amp)
    hist = []
    g.train(); p.train()
    for ep in range(epocas):
        t0, perdidas, aciertos = time.time(), [], []
        for b in torch.randperm(len(X_train), device=device)[:pasos_ep * N].split(N):
            xb = X_train[b]
            vistas = torch.cat([vista(xb, color), vista(xb, color)])          # 2N vistas: [vista 1; vista 2]
            with torch.autocast(device.type, dtype=torch.float16, enabled=amp):
                loss, acierto = info_nce(p(g(vistas)), TAU)                     # u = p(g(x̃))
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
            perdidas.append(loss.item()); aciertos.append(acierto.item())
        hist.append({'perdida': float(np.mean(perdidas)), 'acierto_positivo': float(np.mean(aciertos))})
        print(f"época {ep + 1} · pérdida {hist[-1]['perdida']:.3f} · acierto del positivo "
            f"{hist[-1]['acierto_positivo']:.3f} · {time.time() - t0:.0f} s")
    return hist

torch.manual_seed(0)
g, p = resnet18_cifar().to(device), cabeza().to(device)
with torch.no_grad(), torch.autocast(device.type, dtype=torch.float16, enabled=amp):
    g.eval(); p.eval()
    xb = X_train[:N]
    l0, _ = info_nce(p(g(torch.cat([vista(xb), vista(xb)]))), TAU)
print(f'pérdida inicial {l0.item():.3f} · ln(2N − 1) = {math.log(2 * N - 1):.3f}')
hist_vivo = entrenar(g, p, EPOCAS_VIVO)

El **acierto del positivo** es la fracción de filas de $S$ en que la otra vista es la más parecida entre 1023 candidatos: el azar daría 0.001. Sube rápido porque la tarea pretexto es fácil de empezar a resolver; lo que nos importa no es ese número, sino si $\mathbf h$ sirve para algo que la red nunca vio: las clases.

### $k$-NN: medir sin entrenar nada
Para cada imagen de prueba buscamos las $k = 20$ imágenes de entrenamiento con $\mathbf h$ más parecido (coseno) y votamos su clase (diapositiva *$k$-NN in embedding space*). Las etiquetas aparecen aquí por primera vez. Las imágenes entran **sin aumentar**.

In [ ]:
@torch.no_grad()
def embeddings(g, X, chunk=1000):
    g.eval()
    with torch.autocast(device.type, dtype=torch.float16, enabled=amp):
        H = torch.cat([g(norm(X[i:i + chunk].float() / 255)).float() for i in range(0, len(X), chunk)])
    g.train()
    return H                                                # h = g(x), sin aumentos

@torch.no_grad()
def knn(H_train, H_test, k=20, t=0.07, C=10):
    Htr, Hte = F.normalize(H_train, dim=1), F.normalize(H_test, dim=1)
    ytr = y_train.to(Htr.device)
    pred = []
    for i in range(0, len(Hte), 1000):
        sim, j = (Hte[i:i + 1000] @ Htr.T).topk(k, dim=1)          # los k vecinos más parecidos
        votos = torch.zeros(len(j), C, device=Htr.device).scatter_add_(1, ytr[j], (sim / t).exp())
        pred.append(votos.argmax(1))
    return (torch.cat(pred).cpu() == y_test).float().mean().item()

torch.manual_seed(0)
g_azar = resnet18_cifar().to(device)
acc = {'aleatorio': knn(embeddings(g_azar, X_train), embeddings(g_azar, X_test)),
       f'SimCLR, {EPOCAS_VIVO} épocas': knn(embeddings(g, X_train), embeddings(g, X_test))}
acc

## 4. El *checkpoint* largo
Con más épocas la pérdida sigue bajando y $\mathbf h$ mejora mucho. Ese entrenamiento ya está hecho, y la celda dice cuántas épocas tiene: en Kaggle, agrega el dataset [`juanmartinezv4399/si7011-s04-simclr`](https://www.kaggle.com/datasets/juanmartinezv4399/si7011-s04-simclr) con **Add Input**. En Colab, la celda lo descarga con tu secreto `KAGGLE_API_TOKEN`; en local o en Lightning, pon `simclr_resnet18_cifar10.pt` junto al notebook o en `data/`.
Sin el archivo, el notebook sigue con el modelo de la sección 3.

In [ ]:
CKPT_HANDLE = 'juanmartinezv4399/si7011-s04-simclr'   # los checkpoints largos de SimCLR, con y sin color

def buscar(nombre):
    # orden: carpeta del notebook y data/, input de Kaggle, Lightning, y por último kagglehub con tu token
    for d in [Path('.'), *ROOTS]:
        hits = sorted(d.glob(f'**/{nombre}')) if d.exists() else []
        if hits:
            return hits[0]
    if kaggle_token():
        try:
            import kagglehub
            hits = sorted(Path(kagglehub.dataset_download(CKPT_HANDLE)).glob(f'**/{nombre}'))
            if hits:
                return hits[0]
        except Exception as e:
            print('kagglehub:', type(e).__name__, '· no se pudo bajar', CKPT_HANDLE)
    return None

ruta = buscar('simclr_resnet18_cifar10.pt')

if ruta is not None:
    ck = torch.load(ruta, map_location=device)
    g_largo = resnet18_cifar().to(device); g_largo.load_state_dict(ck['g'])
    p_largo = cabeza().to(device); p_largo.load_state_dict(ck['p'])
    acc[f"SimCLR, {ck['epocas']} épocas"] = knn(embeddings(g_largo, X_train), embeddings(g_largo, X_test))
    plt.figure(figsize=(5, 2.6)); plt.plot([h['perdida'] for h in ck['historia']])
    plt.xlabel('época'); plt.ylabel('InfoNCE'); plt.title(f"{ruta.name}"); plt.show()
else:
    print('No hay checkpoint: las secciones 5 y 6 usan el modelo de 3 épocas.')
    g_largo, p_largo = g, p
acc

Sin una sola etiqueta durante el entrenamiento, un $k$-NN sobre $\mathbf h$ clasifica la mayoría de las imágenes de prueba. Con pesos aleatorios, el mismo $k$-NN queda muy por debajo: lo que cambió no es la arquitectura, es el **objetivo**.

## 5. La matriz de similitudes
Tomamos $N = 8$ imágenes, una por clase, y sus dos vistas. Es la figura de la diapositiva *InfoNCE is a cross-entropy*: cada fila es un softmax sobre las otras 15 vistas, y la respuesta correcta está en la diagonal desplazada.

In [ ]:
torch.manual_seed(5)
x8 = X_train[idx8]
vistas8 = torch.cat([vista(x8), vista(x8)])                       # [vista 1 de las 8; vista 2 de las 8]
with torch.no_grad():
    g_largo.eval(); p_largo.eval()
    u8 = F.normalize(p_largo(g_largo(vistas8)).float(), dim=1)
    g_largo.train(); p_largo.train()
S8 = (u8 @ u8.T / TAU).cpu()
S8.fill_diagonal_(float('nan'))
fig, axes = plt.subplots(1, 2, figsize=(11, 4.3), gridspec_kw={'width_ratios': [1.2, 1]})
im = axes[0].imshow(S8, cmap='RdBu_r', vmin=-1 / TAU, vmax=1 / TAU)
for i in range(16):
    axes[0].add_patch(plt.Rectangle(((i + 8) % 16 - 0.5, i - 0.5), 1, 1, fill=False, ec='orange', lw=2))
etiq = [f'{CLASSES[c][:4]}·{v}' for v in (1, 2) for c in range(8)]
axes[0].set_xticks(range(16), etiq, rotation=90, fontsize=7); axes[0].set_yticks(range(16), etiq, fontsize=7)
axes[0].set_title('$s_{ij}$ (naranja: la otra vista)', fontsize=9); axes[0].grid(False); plt.colorbar(im, ax=axes[0], fraction=0.046)
fila = 3
prob = torch.softmax(torch.nan_to_num(S8[fila], nan=-1e9), 0)
cols = ['tab:orange' if j == (fila + 8) % 16 else '0.6' for j in range(16)]
axes[1].bar(range(16), prob, color=cols); axes[1].set_xticks(range(16), etiq, rotation=90, fontsize=7)
axes[1].set_title(f'fila {fila} como softmax · $L_i$ = {-math.log(prob[(fila + 8) % 16]):.2f}', fontsize=9)
plt.tight_layout(); plt.show()
np.savez_compressed('similitud_n8.npz', S=S8.numpy(), vistas=ver(vistas8).numpy(), clases=np.array(CLASSES[:8]))   # figura S04-F07

## 6. Sin *color jitter*: el atajo del color
**Predice:** si quitamos el *color jitter* y la escala de grises, ¿qué le pasa al acierto del positivo? ¿Y al $k$-NN?

Dos recortes de la misma foto comparten casi el mismo histograma de colores. Sin *color jitter*, la red puede emparejar las vistas mirando solo el color, y no necesita aprender formas. Esta sección compara el *checkpoint* sin color (`simclr_sin_color.pt`, en el mismo dataset de Kaggle) con el de la sección 4.

In [ ]:
ruta_sc = buscar('simclr_sin_color.pt')
if ruta_sc is not None:
    ck_sc = torch.load(ruta_sc, map_location=device)
    g_sc = resnet18_cifar().to(device); g_sc.load_state_dict(ck_sc['g'])
    acc['SimCLR sin color'] = knn(embeddings(g_sc, X_train), embeddings(g_sc, X_test))
    for n, c in [('con color jitter', ck), ('sin color jitter', ck_sc)]:
        h = c['historia'][-1]
        print(f"{n} · pérdida final {h['perdida']:.3f} · acierto del positivo {h['acierto_positivo']:.3f}")
else:
    print(f'No encontré simclr_sin_color.pt: agrega el dataset {CKPT_HANDLE}.')
acc

Sin *color jitter*, la tarea pretexto queda **más fácil**: menor pérdida y mayor acierto del positivo. Y aun así $\mathbf h$ es **peor** para clasificar. La red resolvió el problema que le pusimos, no el que nos interesaba. Las aumentaciones son la única definición de "lo mismo" que recibe la red, y elegirlas es una decisión de diseño, no un detalle.

## Guardar el encoder $g$
Solo se guarda $g$: la cabeza $p$ se usó para entrenar y se descarta (diapositiva *Why a projection head?*). El notebook 5 carga este archivo.

In [ ]:
torch.save({'g': g_largo.state_dict(), 'arquitectura': 'resnet18_cifar', 'tau': TAU, 'N': N,
            'normalizacion': {'mean': MEAN.flatten().tolist(), 'std': STD.flatten().tolist()}}, 'simclr_g.pt')
print('simclr_g.pt guardado ·', {k: round(v, 3) for k, v in acc.items()})

## Tu turno · 15 minutos
1. Con el modelo de 3 épocas, cambia $\tau$ a 0.05 y a 1.0 (`TAU`, sección 3) y entrena 3 épocas cada uno. ¿Cuál baja más rápido? ¿Cuál tiene mejor $k$-NN? Relaciónalo con la animación *Pull and push on the sphere*.
2. Calcula el $k$-NN con $\mathbf u = p(\mathbf h)$ en lugar de $\mathbf h$ (usa `p_largo`). ¿Cuál es mejor? ¿Coincide con la diapositiva *Why a projection head?*
3. Reduce el lote a $N = 64$ con el mismo número de épocas. ¿Qué pasa con la pérdida inicial y con el $k$-NN? (diapositiva *Negatives and batch size*).

**Para pensar.** El recorte aleatorio puede dejar en una vista solo el cielo y en la otra solo el avión. ¿Qué le enseña eso a la red? ¿Cuándo sería un problema para tu dominio?

In [ ]:
# Experimento 1: TAU = 0.05 y TAU = 1.0, 3 épocas cada uno

### Referencias
- Chen, Kornblith, Norouzi y Hinton (2020), *A Simple Framework for Contrastive Learning of Visual Representations*, ICML (SimCLR; el atajo del color está en su sección 3).
- van den Oord, Li y Vinyals (2018), *Representation Learning with Contrastive Predictive Coding*, arXiv:1807.03748 (InfoNCE).
- Wu, Xiong, Yu y Lin (2018), *Unsupervised Feature Learning via Non-Parametric Instance Discrimination*, CVPR (el $k$-NN ponderado de la sección 3).
- El material de la versión anterior del curso (`Lecture03-04/notebooks/4_self-supervised`) inspiró la estructura de las dos vistas y la métrica del positivo.